## Required libraries installation

In [ ]:
%pip install accelerate scipy librosa -q
%pip install transformers torchaudio pandas datasets -q
%pip install soundfile -q
%pip install transformers==5.18.0 -q
%pip install tqdm -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## All imports and custom built modules and functions

In [6]:
# Normal imports
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import gc

# Torch imports
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor


## Device agnostic code

In [1]:
! nvidia-smi

Fri Oct 16 23:55:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.04              KMD Version: 610.57.04     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   41C    P8              2W /  115W |       5MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
device = "cuda" if torch.cuda.is_available else "cpu"
device
# if it shows cpu download cuda torch audio

'cuda'

In [9]:
! python preprocessing_pipeline.py

      EcoAST TRAIN / VAL / TEST PIPELINE          
Indexing raw audio files for FSD50K mapping...
Collected total of 2826 raw audio records.

Standardizing audio files for split: 'TRAIN' (1978 samples)...
Processing train: 100% 1978/1978 [00:52<00:00, 37.87it/s]
Exported 1978 rows to metadata: split_dataset/train_metadata.csv

Standardizing audio files for split: 'VAL' (424 samples)...
Processing val: 100% 424/424 [00:15<00:00, 26.67it/s]
Exported 424 rows to metadata: split_dataset/val_metadata.csv

Standardizing audio files for split: 'TEST' (424 samples)...
Processing test: 100% 424/424 [00:09<00:00, 43.47it/s]
Exported 424 rows to metadata: split_dataset/test_metadata.csv

             PIPELINE EXPORT COMPLETE             


In [13]:
from preprocessing_pipeline import get_dataloaders

from training import train_ast_model, ForestThreatAST, AST_VARIANTS

NUM_CLASSES = 4  # chainsaw, gunshot, engine, background
BATCH_SIZE = 2
EPOCHS = 5
TARGET_CLASSES = ["chainsaw", "gunshot", "engine", "background"]

# Training AST-Audioset-10x10

In [ ]:
print(f"Starting Training...")
model_v1 = train_ast_model(
    model=model_v1,
    train_loader=train_loader_v1,
    device=device,
    model_name=variant_name,
    num_epochs=EPOCHS,
    save_path=f"trained_models/ecoast_{variant_name}.pth",
)

Loading DataLoaders for AST-AudioSet-10x10...
Initializing AST-AudioSet-10x10...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Starting Training...

Training [AST-AudioSet-10x10] on CUDA | TensorBoard: runs/AST-AudioSet-10x10


[AST-AudioSet-10x10] Epoch [1/5]:   0%|          | 0/989 [00:00<?, ?it/s]

 -> Epoch 1 Summary | Loss: 0.1318 | Acc: 94.87% | Precision: 0.6762 | mAP: 0.9439


...

 -> Epoch 3 Summary | Loss: 0.0420 | Acc: 98.50% | Precision: 0.7258 | mAP: 0.9928


[AST-AudioSet-10x10] Epoch [4/5]:   0%|          | 0/989 [00:00<?, ?it/s]

 -> Epoch 4 Summary | Loss: 0.0314 | Acc: 98.93% | Precision: 0.7329 | mAP: 0.9969


[AST-AudioSet-10x10] Epoch [5/5]:   0%|          | 0/989 [00:00<?, ?it/s]

 -> Epoch 5 Summary | Loss: 0.0162 | Acc: 99.48% | Precision: 0.7423 | mAP: 0.9991
Checkpoint saved: 'trained_models/ecoast_AST-AudioSet-10x10.pth'


In [ ]:
# %% [Launch TensorBoard Background Server & Print Clickable Browser Link]
import subprocess
import time
from IPython.display import display, HTML

PORT = 6006
LOG_DIR = "runs"

# Kill any existing TensorBoard process occupying the port so it starts cleanly
subprocess.run(
    ["pkill", "-f", f"tensorboard.*--port {PORT}"], stderr=subprocess.DEVNULL
)

# Start TensorBoard in the background bound to localhost
tb_process = subprocess.Popen(
    ["tensorboard", "--logdir", LOG_DIR, "--host", "localhost", "--port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

time.sleep(2)  # Give the server 2 seconds to spin up

url = f"http://localhost:{PORT}"
print(f"TensorBoard is live at: {url}")
display(
    HTML(
        f'<h3>🚀 <a href="{url}" target="_blank">Click here to open TensorBoard in your browser ({url})</a></h3>'
    )
)

TensorBoard is live at: http://localhost:6006


In [ ]:
import torch
import torch.nn as nn
from transformers import ASTConfig, ASTForAudioClassification

# Registry of the 3 AST Variants required by the class
AST_VARIANTS = {
    "AST-AudioSet-10x10": {
        "checkpoint": "MIT/ast-finetuned-audioset-10-10-0.4593",
        "frequency_stride": 10,
        "time_stride": 10,
    },
    "AST-SpeechCommands-V2": {
        "checkpoint": "MIT/ast-finetuned-speech-commands-v2",
        "frequency_stride": 10,
        "time_stride": 10,
    },
    "AST-FastStride-16x16": {
        "checkpoint": "MIT/ast-finetuned-audioset-10-10-0.4593",
        "frequency_stride": 16,
        "time_stride": 16,
    },
}


class ForestThreatAST(nn.Module):
    """
    AI Systems Design Layer 4 & 5:
    Configurable Audio Spectrogram Transformer Backbone
    """

    def __init__(self, num_classes: int, variant_name: str = "AST-AudioSet-10x10"):
        super().__init__()
        if variant_name not in AST_VARIANTS:
            raise ValueError(
                f"Unknown variant '{variant_name}'. Choose from: {list(AST_VARIANTS.keys())}"
            )

        self.variant_name = variant_name
        spec = AST_VARIANTS[variant_name]

        # Configure architecture (enforces 1024 frames & custom patch strides)
        config = ASTConfig.from_pretrained(
            spec["checkpoint"],
            max_length=1024,
            num_mel_bins=128,
            frequency_stride=spec["frequency_stride"],
            time_stride=spec["time_stride"],
        )

        # ignore_mismatched_sizes=True interpolates positional embeddings for SpeechCommands & 16x16 stride
        self.ast = ASTForAudioClassification.from_pretrained(
            spec["checkpoint"],
            config=config,
            ignore_mismatched_sizes=True,
        )

        # Swap the default classification head for our multi-label target classes
        hidden_size = self.ast.classifier.dense.in_features
        self.ast.classifier.dense = nn.Linear(hidden_size, num_classes)

    def forward(self, input_values):
        return self.ast(input_values=input_values).logits

In [ ]:
# 1. Define hardware and classes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_CLASSES = ["chainsaw", "gunshot", "engine", "background"]

# 2. Initialize the architecture (Make sure to match the variant_name you trained on Colab)
model = ForestThreatAST(
    num_classes=len(TARGET_CLASSES), variant_name="AST-AudioSet-10x10"
).to(device)

# 3. Load the downloaded Colab weights safely
model_path = "/home/warden/dl_project/trained_models/eco_stat_ast_10_10.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model successfully loaded and ready for local inference!")

/tmp/ipykernel_105722/3437552710.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


RuntimeError: Error(s) in loading state_dict for ForestThreatAST:
	Missing key(s) in state_dict: "ast.audio_spectrogram_transformer.encoder.layer.0.attention.attention.query.weight", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.attention.query.bias", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.attention.key.weight", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.attention.key.bias", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.attention.value.weight", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.attention.value.bias", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.output.dense.weight", "ast.audio_spectrogram_transformer.encoder.layer.0.attention.output.dense.bias", "ast.audio_spectrogram_transformer.encoder.layer.0.intermediate.dense.weight", "ast.audio_spectrogram_transformer.encoder.layer.0.intermediate.dense.bias", "ast.audio_spectrogram_transformer.encoder.layer.0.output.dense.weight", "ast

# Training AST-SpeechCommands-V2

In [ ]:
# %% [Cell B1: Train Model 2 (AST-SpeechCommands-V2)]
if device == "cuda":
    torch.cuda.empty_cache()

variant_name_2 = "AST-SpeechCommands-V2"
save_path_2 = "trained_models/ast_speechcommands_v2.pth"

print(f"Initializing {variant_name_2}...")
model_2 = ForestThreatAST(
    num_classes=len(TARGET_CLASSES), variant_name=variant_name_2
).to(device)

# Train and log to runs/AST-SpeechCommands-V2
train_ast_model(
    model=model_2,
    train_loader=train_loader,
    device=device,
    model_name=variant_name_2,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    save_path=save_path_2,
    log_dir="runs",
)

# Free VRAM after training completes
del model_2
if device == "cuda":
    torch.cuda.empty_cache()

# Training AST-FastStride-16x16

In [ ]:
# %% [Cell B2: Train Model 3 (AST-FastStride-16x16)]
if device == "cuda":
    torch.cuda.empty_cache()

variant_name_3 = "AST-FastStride-16x16"
save_path_3 = "trained_models/ast_faststride_16x16.pth"

print(f"Initializing {variant_name_3}...")
model_3 = ForestThreatAST(
    num_classes=len(TARGET_CLASSES), variant_name=variant_name_3
).to(device)

# Train and log to runs/AST-FastStride-16x16
train_ast_model(
    model=model_3,
    train_loader=train_loader,
    device=device,
    model_name=variant_name_3,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    save_path=save_path_3,
    log_dir="runs",
)

# Free VRAM after training completes
del model_3
if device == "cuda":
    torch.cuda.empty_cache()

print("\nAll 3 AST models are now trained and logged in the 'runs/' folder!")